# Twitter Signals vs Price

Explores tweet mention spikes from imported KOL/trader data against
daily OHLCV. Requires:

1. `uv run ccquant sync tweets` (or import fixture into inbox)
2. `uv run dbt build --project-dir dbt --profiles-dir dbt`

In [1]:
import os
from pathlib import Path

import plotly.express as px

from ccquant.forecasting import load_signals_panel, load_tweet_panel

# Notebooks often start with cwd=notebooks/; resolve DB from repo root.
_root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DB = Path(os.environ.get("CCQUANT_DB", _root / "data" / "ccquant.duckdb")).resolve()
signals = load_signals_panel(DB)
tweets = load_tweet_panel(DB)
signals.head(), tweets.head()

(shape: (5, 44)
 ┌────────┬────────────┬───────┬────────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
 │ symbol ┆ date       ┆ open  ┆ high   ┆ … ┆ insider_clu ┆ tweet_menti ┆ kol_tweet_m ┆ tweet_senti │
 │ ---    ┆ ---        ┆ ---   ┆ ---    ┆   ┆ ster_count  ┆ on_count    ┆ ention_coun ┆ ment_net    │
 │ str    ┆ date       ┆ f64   ┆ f64    ┆   ┆ ---         ┆ ---         ┆ t           ┆ ---         │
 │        ┆            ┆       ┆        ┆   ┆ i64         ┆ i32         ┆ ---         ┆ i32         │
 │        ┆            ┆       ┆        ┆   ┆             ┆             ┆ i32         ┆             │
 ╞════════╪════════════╪═══════╪════════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
 │ AAVE   ┆ 2026-07-11 ┆ 95.7  ┆ 102.17 ┆ … ┆ null        ┆ null        ┆ null        ┆ null        │
 │ AAVE   ┆ 2026-07-12 ┆ 98.02 ┆ 100.99 ┆ … ┆ null        ┆ null        ┆ null        ┆ null        │
 │ AAVE   ┆ 2026-07-13 ┆ 96.8  ┆ 97.84  ┆ … ┆ null        ┆ null  

In [2]:
symbol = "SOL"
panel = signals.filter(signals["symbol"] == symbol).select(
    ["date", "close", "tweet_mention_count", "tweet_sentiment_net"]
)
fig = px.line(panel, x="date", y=["close", "tweet_mention_count"], title=f"{symbol} price vs tweet mentions")
fig.show()